In [ ]:
!nvidia-smi

Fri May  1 01:31:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart pillow
!pip install -q transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 17.2 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import fastapi

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("FastAPI:", fastapi.__version__)

Torch: 2.10.0+cu128
CUDA available: True
Transformers: 5.0.0
FastAPI: 0.136.1


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "pqthinh232/HCMUS-Qwen2-VL-2B-Instruct-Vietnamese-Image-Captioning-for-blind-E2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=128 * 28 * 28,
    max_pixels=384 * 28 * 28
)

model.eval()

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 01083.jpg to 01083.jpg


In [ ]:
from PIL import Image
from qwen_vl_utils import process_vision_info
import time

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

prompt = (
    "Viết đúng một câu ngắn bằng tiếng Việt, mô tả vật cản hoặc tình huống nguy hiểm chính "
    "trong ảnh và đưa ra hướng dẫn di chuyển an toàn cho người khiếm thị. "
    "Không giải thích thêm."
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {
                "type": "text",
                "text": prompt,
            },
        ],
    }
]

start = time.time()

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

generated_ids_trimmed = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

end = time.time()

print("Caption:", output_text)
print("Inference time:", round(end - start, 3), "seconds")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Caption: lòng đường phía trước có xe máy đang lưu thông dưới lòng đường, vỉa hè bên phải sát các tòa nhà có người đứng chờ nên bạn cần dò gậy lách qua
Inference time: 5.453 seconds


In [ ]:
import io
import time
import torch
import nest_asyncio
import uvicorn

from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()

app = FastAPI(
    title="Blind Assistant Backend",
    description="API nhận ảnh và trả mô tả tiếng Việt cho người khiếm thị",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DEFAULT_PROMPT = (
    "Viết đúng một câu ngắn bằng tiếng Việt, mô tả vật cản hoặc tình huống nguy hiểm chính "
    "trong ảnh và đưa ra hướng dẫn di chuyển an toàn cho người khiếm thị. "
    "Không giải thích thêm."
)


def resize_image(image: Image.Image, max_side: int = 768) -> Image.Image:
    width, height = image.size

    if max(width, height) <= max_side:
        return image

    scale = max_side / max(width, height)
    new_width = int(width * scale)
    new_height = int(height * scale)

    return image.resize((new_width, new_height))


def generate_caption(image: Image.Image, prompt: str = DEFAULT_PROMPT) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text.strip()


@app.get("/")
def root():
    return {
        "message": "Blind Assistant Backend is running",
        "model": MODEL_ID
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "cuda": torch.cuda.is_available(),
        "device": device,
        "model_loaded": model is not None
    }


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    start_time = time.time()

    if file.content_type is None or not file.content_type.startswith("image/"):
        raise HTTPException(
            status_code=400,
            detail="File gửi lên phải là ảnh."
        )

    try:
        image_bytes = await file.read()

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        image = resize_image(image, max_side=768)

        caption = generate_caption(image)
        if not caption:
            caption = "Không thể xác định rõ vật cản trong ảnh, bạn hãy di chuyển chậm và kiểm tra xung quanh."

        latency = time.time() - start_time

        return {
            "success": True,
            "caption": caption,
            "latency_seconds": round(latency, 3)
        }

    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=str(error)
        )

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn

nest_asyncio.apply()

NGROK_AUTH_TOKEN = "3D4dVO5QKv79kLD38sVmmd0sqMP_2ZtoQ9Wv2fyo3nHzSXdJC"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000).public_url

print("Public API URL:", public_url)
print("Health check:", public_url + "/health")
print("Predict endpoint:", public_url + "/predict")

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()

Public API URL: https://buckshot-marshy-delicacy.ngrok-free.dev
Health check: https://buckshot-marshy-delicacy.ngrok-free.dev/health
Predict endpoint: https://buckshot-marshy-delicacy.ngrok-free.dev/predict


INFO:     Started server process [1169]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     54.86.50.139:0 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     54.86.50.139:0 - "POST /predict?Key=File HTTP/1.1" 422 Unprocessable Entity
INFO:     171.243.49.247:0 - "GET /health HTTP/1.1" 200 OK
INFO:     54.86.50.139:0 - "POST /predict?Key=File HTTP/1.1" 200 OK
INFO:     54.86.50.139:0 - "POST /predict?Key=File HTTP/1.1" 200 OK
INFO:     54.86.50.139:0 - "POST /predict?Key=File HTTP/1.1" 200 OK
